# Stage 4 · Reward Design & Shaping — SOLUTION
### Topics: Rule-Based vs Learned Rewards · Reward Hacking · PRMs · ORMs · Format Rewards · Verifiable Rewards


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import re
import json
import math
from typing import List, Tuple, Optional, Dict, Callable
from dataclasses import dataclass, field


---
## 1 · Verifiable Rewards — Math & Code

The cleanest RL signal comes from tasks where correctness is **objectively checkable**:
- **Math:** compare final numerical answer to ground truth
- **Code:** run tests, check output matches expected
- **Logic/formal:** automated theorem provers

### Why verifiable rewards are so powerful (DeepSeek-R1 insight)
- No reward model needed → no reward hacking via RM exploitation
- The reward is perfectly calibrated — no distribution shift
- Scales naturally: harder problems are still verifiable
- Enables long chain-of-thought: intermediate steps can be arbitrary as long as the answer is correct

### Reward design choices
| Design | Signal density | Risk |
|---|---|---|
| Binary (correct/wrong) | Very sparse | Hard to learn; no gradient on wrong answers |
| Partial credit (edit distance) | Dense | May reward near-misses that don't generalise |
| Format + correctness | Medium | Format hacking (correct format, wrong content) |
| Process reward (step-by-step) | Dense | Expensive to label; PRM training needed |


In [3]:
def math_answer_reward(
    prediction: str,
    ground_truth: str,
    partial_credit: bool = False,
) -> float:
    """
    Extract final numeric answer from a math solution string and compare.
    Handles common formats: '#### 42', 'The answer is 42', '= 42'.
    Returns 1.0 for exact match, 0.0 otherwise (or partial credit via edit distance).
    """
    def extract_number(text: str) -> Optional[float]:
        # Match '#### N', '= N', last number in text
        patterns = [r'####\s*([+-]?[\d,]+\.?\d*)', r'=\s*([+-]?[\d,]+\.?\d*)', r'([+-]?[\d,]+\.?\d*)']
        for pat in patterns:
            m = re.search(pat, text)
            if m:
                try:
                    return float(m.group(1).replace(',', ''))
                except ValueError:
                    continue
        return None

    pred_num = extract_number(prediction)
    true_num = extract_number(ground_truth)

    if pred_num is None or true_num is None:
        return 0.0

    if abs(pred_num - true_num) < 1e-5:
        return 1.0

    if partial_credit:
        # Reward based on relative error (diminishes quickly)
        rel_err = abs(pred_num - true_num) / (abs(true_num) + 1e-8)
        return max(0.0, 1.0 - rel_err)

    return 0.0


def code_execution_reward(
    code_str: str,
    test_cases: List[Tuple[str, str]],   # List of (input_str, expected_output)
    timeout: float = 1.0,
) -> float:
    """
    Run code against test cases, return fraction passed.
    Safe eval in restricted namespace.
    Returns reward in [0, 1].
    """
    passed = 0
    for input_str, expected in test_cases:
        try:
            namespace = {}
            exec(code_str, namespace)
            # Expect a function called 'solution'
            if 'solution' not in namespace:
                continue
            result = str(namespace['solution'](input_str))
            if result.strip() == expected.strip():
                passed += 1
        except Exception:
            pass
    return passed / max(len(test_cases), 1)


def format_reward(
    text: str,
    required_tags: List[str] = ("<think>", "</think>", "<answer>", "</answer>"),
) -> float:
    """
    Check that text contains all required structural tags in order.
    Used to reward chain-of-thought format compliance.
    Returns 1.0 if all tags present and in order, 0.0 otherwise.
    """
    pos = 0
    for tag in required_tags:
        idx = text.find(tag, pos)
        if idx == -1:
            return 0.0
        pos = idx + len(tag)
    return 1.0


def composite_reward(
    prediction: str,
    ground_truth: str,
    test_cases: Optional[List[Tuple[str, str]]] = None,
    w_correctness: float = 1.0,
    w_format:      float = 0.2,
) -> Tuple[float, Dict[str, float]]:
    """
    Weighted combination of correctness + format rewards.
    Used in DeepSeek-R1 and similar pipelines.
    """
    r_correct = math_answer_reward(prediction, ground_truth)
    r_format  = format_reward(prediction)
    r_total   = w_correctness * r_correct + w_format * r_format
    return r_total, {"correctness": r_correct, "format": r_format, "total": r_total}


# ── Sanity checks ─────────────────────────────────────────────────────────
# Math reward
assert math_answer_reward("The answer is #### 42", "#### 42") == 1.0
assert math_answer_reward("#### 42", "#### 43") == 0.0
assert math_answer_reward("I think the answer is 3.14", "= 3.14") == 1.0
assert math_answer_reward("no number here", "#### 5") == 0.0

# Format reward
good_cot = "<think>Let me solve this step by step...</think><answer>42</answer>"
bad_cot  = "The answer is 42"
assert format_reward(good_cot) == 1.0
assert format_reward(bad_cot)  == 0.0

# Composite
r, info = composite_reward(good_cot + " #### 42", "#### 42")
print(f"math_answer_reward ✓")
print(f"format_reward      ✓")
print(f"composite_reward   ✓  {info}")

# Code reward
code = """
def solution(x):
    return str(int(x) * 2)
"""
reward = code_execution_reward(code, [("3", "6"), ("5", "10"), ("0", "0")])
assert reward == 1.0, f"Expected 1.0, got {reward}"
print(f"code_execution_reward ✓  passed all tests")


math_answer_reward ✓
format_reward      ✓
composite_reward   ✓  {'correctness': 1.0, 'format': 1.0, 'total': 1.2}
code_execution_reward ✓  passed all tests


---
## 2 · Outcome Reward Models (ORMs) vs Learned Reward Models

When verifiable rewards aren't available, we train a **reward model** (RM) on human preferences.

### Outcome Reward Model (ORM)
- Scores the **complete response** holistically
- Input: (prompt, full_response) → scalar
- Simple but sparse signal — no feedback on intermediate steps
- Used in InstructGPT, most production RLHF systems

### Reward model failure modes
1. **Out-of-distribution:** RM trained on distribution A, policy generates distribution B → unreliable scores
2. **Length bias:** RM often scores longer responses higher regardless of quality
3. **Style hacking:** model learns RM prefers certain words/phrases independent of content
4. **Overoptimisation:** Gao et al. (2023) showed RM score peaks then declines relative to ground truth

### Reward normalisation
Raw RM scores should be normalised before use as RL rewards:
- **Z-score:** `r_norm = (r - μ) / σ` — standardise across batch
- **Percentile:** `r_norm = rank(r) / n` — robust to outliers
- **Clipping:** `r_norm = clip(r, r_min, r_max)` — prevents extreme values from dominating


In [4]:
class OutcomeRewardModel(nn.Module):
    """
    Simplified ORM: takes (prompt_emb, response_emb) → scalar reward.
    In production: LM backbone with linear reward head on [EOS] token.
    """
    def __init__(self, embed_dim: int = 64, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1),
        )
        self.embed_dim = embed_dim

    def forward(self, prompt_emb: torch.Tensor, response_emb: torch.Tensor) -> torch.Tensor:
        x = torch.cat([prompt_emb, response_emb], dim=-1)   # (B, 2D)
        return self.net(x).squeeze(-1)                       # (B,)


def normalize_rewards_zscore(rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Z-score normalisation: (r - μ) / σ"""
    return (rewards - rewards.mean()) / (rewards.std() + eps)


def normalize_rewards_percentile(rewards: torch.Tensor) -> torch.Tensor:
    """Rank-based normalisation: rank / n, in [1/n, 1]"""
    n      = rewards.shape[0]
    ranks  = torch.argsort(torch.argsort(rewards)) + 1    # 1-indexed ranks
    return ranks.float() / n


def clip_rewards(
    rewards: torch.Tensor,
    percentile_low:  float = 5.0,
    percentile_high: float = 95.0,
) -> torch.Tensor:
    """Clip rewards at given percentiles to remove outlier rewards."""
    low  = torch.quantile(rewards, percentile_low  / 100.0)
    high = torch.quantile(rewards, percentile_high / 100.0)
    return torch.clamp(rewards, low, high)


def detect_length_bias(
    rewards:    torch.Tensor,  # (B,)
    lengths:    torch.Tensor,  # (B,) — response lengths in tokens
    threshold:  float = 0.3,   # Pearson correlation threshold for bias
) -> Tuple[float, bool]:
    """
    Compute Pearson correlation between reward and response length.
    High positive correlation → RM has length bias.
    Returns (correlation, is_biased).
    """
    r = rewards.float();  l = lengths.float()
    r_z = (r - r.mean()) / (r.std() + 1e-8)
    l_z = (l - l.mean()) / (l.std() + 1e-8)
    corr = (r_z * l_z).mean().item()
    return corr, abs(corr) > threshold


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B, D = 16, 64
orm  = OutcomeRewardModel(D)
p_emb = torch.randn(B, D)
r_emb = torch.randn(B, D)
rewards = orm(p_emb, r_emb)
assert rewards.shape == (B,)

# Z-score: mean≈0, std≈1
r_z = normalize_rewards_zscore(rewards.detach())
assert abs(r_z.mean().item()) < 1e-5
assert abs(r_z.std().item()  - 1.0) < 1e-5

# Percentile: values in (0, 1]
r_pct = normalize_rewards_percentile(rewards.detach())
assert r_pct.min().item() > 0 and r_pct.max().item() <= 1.0

# Length bias: synthetic biased rewards (reward = length + noise)
lengths = torch.randint(50, 500, (B,)).float()
biased_rewards = lengths + torch.randn(B) * 5
corr, is_biased = detect_length_bias(biased_rewards, lengths)
assert is_biased, f"Expected length bias, corr={corr:.3f}"

unbiased_rewards = torch.randn(B)
corr2, is_biased2 = detect_length_bias(unbiased_rewards, lengths)
assert not is_biased2, f"Unexpected bias, corr={corr2:.3f}"

print(f"OutcomeRewardModel      ✓  output shape: {rewards.shape}")
print(f"normalize_rewards_zscore ✓  mean={r_z.mean().item():.5f} std={r_z.std().item():.5f}")
print(f"normalize_rewards_percentile ✓  range=[{r_pct.min().item():.3f}, {r_pct.max().item():.3f}]")
print(f"detect_length_bias      ✓  biased={is_biased} (corr={corr:.3f})  unbiased={is_biased2}")


OutcomeRewardModel      ✓  output shape: torch.Size([16])
normalize_rewards_zscore ✓  mean=-0.00000 std=1.00000
normalize_rewards_percentile ✓  range=[0.062, 1.000]
detect_length_bias      ✓  biased=True (corr=0.937)  unbiased=False


---
## 3 · Process Reward Models (PRMs)

**PRMs (Lightman et al., 2023 "Let's Verify Step by Step")** reward each **reasoning step** separately,  
not just the final answer.

### Motivation
- Math reasoning chains can be long (10–30 steps)
- ORM: only knows if the final answer is right — no gradient on intermediate steps
- PRM: tells the model *which step went wrong* — much denser signal

### PRM architecture
Same backbone as ORM, but applied at each **step boundary** (newline or `\n\n`):
```
prompt → step_1 → [PRM score_1] → step_2 → [PRM score_2] → ... → answer → [PRM score_T]
```

### Training PRMs (Monte Carlo estimation)
Without human labels on every step, use MC rollouts to estimate step quality:
1. Sample K completions from each step onwards
2. Step quality = fraction of completions that reach the correct answer
3. This is the **process reward** for that step

### Combining PRM with RL
The per-step PRM reward replaces (or augments) the terminal ORM reward in the RL loop.  
This gives a much denser training signal, especially for long chains.

### PRM vs ORM comparison
| | ORM | PRM |
|---|---|---|
| Reward density | Terminal only | Per step |
| Training data | Preference pairs | Step-level labels or MC |
| Signal quality | Coarse | Fine-grained |
| Used in | InstructGPT, most RLHF | OpenAI o1, DeepSeek-R1 |


In [5]:
class ProcessRewardModel(nn.Module):
    """
    PRM: scores each reasoning step independently.
    In practice: LM backbone, linear head applied at each step boundary.
    Here: MLP applied to step embeddings.
    """
    def __init__(self, embed_dim: int = 64, hidden: int = 128):
        super().__init__()
        self.step_scorer = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden),  # (prompt_emb, step_emb)
            nn.GELU(),
            nn.Linear(hidden, 1),
        )

    def score_steps(
        self,
        prompt_emb:     torch.Tensor,  # (D,) or (1, D)
        step_embs:      torch.Tensor,  # (T_steps, D)
    ) -> torch.Tensor:                 # (T_steps,) per-step scores
        if prompt_emb.dim() == 1:
            prompt_emb = prompt_emb.unsqueeze(0)   # (1, D)
        prompt_rep = prompt_emb.expand(step_embs.shape[0], -1)   # (T_steps, D)
        x = torch.cat([prompt_rep, step_embs], dim=-1)
        return self.step_scorer(x).squeeze(-1)


def mc_process_reward_estimation(
    completion_rewards: torch.Tensor,   # (K,) — 0/1 outcomes of K rollouts from this step
) -> float:
    """
    Monte Carlo estimate of step quality = fraction of rollouts that succeeded.
    This is the training target for the PRM at this step.
    """
    return completion_rewards.float().mean().item()


def aggregate_prm_rewards(
    step_scores: torch.Tensor,   # (T_steps,) — per-step PRM scores
    agg: str = "min",            # 'min', 'mean', 'last', 'product'
) -> float:
    """
    Aggregate per-step PRM scores into a single sequence reward.
    Different aggregations emphasise different properties:
    - 'min': conservative — penalise any bad step (used in Best-of-N selection)
    - 'mean': balanced
    - 'last': only final step matters (approaches ORM)
    - 'product': probability interpretation (score_t ∈ [0,1])
    """
    if agg == "min":
        return step_scores.min().item()
    elif agg == "mean":
        return step_scores.mean().item()
    elif agg == "last":
        return step_scores[-1].item()
    elif agg == "product":
        return step_scores.clamp(0, 1).prod().item()
    else:
        raise ValueError(f"Unknown aggregation: {agg}")


def prm_weighted_reward(
    step_scores:  torch.Tensor,   # (T_steps,) — PRM scores per step
    step_rewards: torch.Tensor,   # (T_steps,) — per-step rewards (e.g., 0 for all, 1 at end)
    discount:     float = 0.99,
) -> torch.Tensor:                # (T_steps,) — PRM-augmented per-step rewards
    """
    Augment sparse rewards with PRM signal:
    r_aug_t = r_t + α * (prm_t - prm_{t-1})
    where α = prm_t improvement is the reward shaping term.
    This is potential-based reward shaping — preserves the optimal policy.
    """
    prm_diff = torch.diff(step_scores, prepend=torch.tensor([0.0]))  # (T_steps,)
    return step_rewards + prm_diff


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
D, T_steps = 64, 5
prm = ProcessRewardModel(D)

prompt_emb = torch.randn(D)
step_embs  = torch.randn(T_steps, D)
scores     = prm.score_steps(prompt_emb, step_embs)
assert scores.shape == (T_steps,), f"Expected ({T_steps},), got {scores.shape}"

# MC estimation: 7 of 10 rollouts succeed → step quality = 0.7
mc_reward = mc_process_reward_estimation(torch.tensor([1,1,1,0,1,1,0,1,1,0]))
assert abs(mc_reward - 0.7) < 1e-5

# Aggregation modes
scores_good = torch.tensor([0.8, 0.9, 0.85, 0.7, 0.95])
scores_bad  = torch.tensor([0.8, 0.9, 0.1,  0.7, 0.95])   # one bad step

min_good = aggregate_prm_rewards(scores_good, "min")
min_bad  = aggregate_prm_rewards(scores_bad,  "min")
assert min_bad < min_good, "Min aggregation should penalise the bad step"

# Reward shaping
step_r   = torch.zeros(T_steps); step_r[-1] = 1.0   # terminal reward only
aug_r    = prm_weighted_reward(scores_good, step_r)
assert aug_r.shape == (T_steps,)

print(f"ProcessRewardModel    ✓  step scores: {scores.tolist()}")
print(f"mc_process_reward     ✓  0.7 = {mc_reward:.1f}")
print(f"aggregate_prm_rewards ✓  min(good)={min_good:.3f}  min(bad)={min_bad:.3f}")
print(f"prm_weighted_reward   ✓  aug_r: {aug_r.tolist()}")


ProcessRewardModel    ✓  step scores: [-0.133689284324646, 0.032155297696590424, 0.004382826387882233, -0.009354688227176666, 0.16657036542892456]
mc_process_reward     ✓  0.7 = 0.7
aggregate_prm_rewards ✓  min(good)=0.700  min(bad)=0.100
prm_weighted_reward   ✓  aug_r: [0.800000011920929, 0.09999996423721313, -0.04999995231628418, -0.15000003576278687, 1.25]


---
## 4 · Reward Shaping, Anti-Hacking Defences & Reward Ensembles

### Reward Shaping (Ng et al., 1999)
Any reward of the form $r'(s,a,s') = r(s,a,s') + \gamma \Phi(s') - \Phi(s)$  
**preserves the optimal policy** for any potential function $\Phi$.

This means we can add structure to the reward signal without changing the optimal behaviour:
- Use PRM scores as $\Phi$ → adds step-level guidance without biasing the optimum
- Use KL as $\Phi$ → equivalent to the KL-penalised objective in RLHF

### Reward Ensembles (Coste et al., 2023)
Train **multiple independent RMs** on different subsets of preference data.  
Use the **minimum** across the ensemble as the actual reward:
$$r_{ensemble}(x,y) = \min_i r_i(x,y)$$

This is a **pessimistic** reward — conservative under uncertainty.  
Prevents the policy from exploiting any single RM's blind spots.

### Length penalty
A common form of reward hacking is **length exploitation** (longer = higher RM score).  
Simple fix: subtract a length penalty from the reward:
$$r_{adj}(x,y) = r(x,y) - \alpha \cdot \max(0, |y| - L_{target})$$

### Reward clipping
Hard clip prevents extreme reward values from causing large gradient updates:
$$r_{clipped} = \text{clip}(r, r_{min}, r_{max})$$


In [6]:
def ensemble_reward(
    rewards: torch.Tensor,   # (B, K) — K reward model scores per completion
    strategy: str = "min",   # 'min', 'mean', 'softmin'
    temperature: float = 1.0,
) -> torch.Tensor:           # (B,)
    """
    Aggregate K reward models into one scalar per completion.
    - 'min': pessimistic (most conservative, best anti-hacking)
    - 'mean': average ensemble
    - 'softmin': soft approximation of min; temperature controls sharpness
    """
    if strategy == "min":
        return rewards.min(dim=1).values
    elif strategy == "mean":
        return rewards.mean(dim=1)
    elif strategy == "softmin":
        # Softmin: weighted average biased toward minimum
        weights = F.softmax(-rewards / temperature, dim=1)
        return (weights * rewards).sum(dim=1)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")


def length_penalized_reward(
    rewards:      torch.Tensor,  # (B,)
    lengths:      torch.Tensor,  # (B,) — completion lengths in tokens
    target_len:   int   = 256,
    penalty_coef: float = 0.001,
) -> torch.Tensor:
    """
    r_adj = r - penalty_coef * max(0, len - target_len)
    Penalises completions longer than target_len.
    """
    excess = (lengths.float() - target_len).clamp(min=0)
    return rewards - penalty_coef * excess


def reward_with_confidence(
    rewards:   torch.Tensor,   # (B, K) — K RM scores
    conf_threshold: float = 0.5,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Return (mean_reward, high_confidence_mask).
    high_confidence = std across ensemble < threshold.
    Don't use low-confidence rewards for training.
    """
    mean_r = rewards.mean(dim=1)
    std_r  = rewards.std(dim=1)
    high_conf = (std_r < conf_threshold)
    return mean_r, high_conf


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B, K = 16, 4
rewards_k = torch.randn(B, K)   # (B, K) ensemble scores

r_min  = ensemble_reward(rewards_k, "min")
r_mean = ensemble_reward(rewards_k, "mean")
r_soft = ensemble_reward(rewards_k, "softmin", temperature=0.5)

assert r_min.shape  == (B,)
assert r_mean.shape == (B,)
assert (r_min <= r_mean).all(), "Min should be ≤ mean"

lengths = torch.randint(100, 512, (B,))
r_raw   = torch.rand(B)
r_adj   = length_penalized_reward(r_raw, lengths, target_len=256, penalty_coef=0.001)
# Short completions: no penalty
short_mask = (lengths <= 256)
assert torch.allclose(r_adj[short_mask], r_raw[short_mask]), "Short completions should not be penalised"
# Long completions: penalised
long_mask = (lengths > 256)
assert (r_adj[long_mask] < r_raw[long_mask]).all(), "Long completions should be penalised"

r_conf, conf_mask = reward_with_confidence(rewards_k, conf_threshold=1.0)
print(f"ensemble_reward         ✓  min ≤ mean: {(r_min <= r_mean).all().item()}")
print(f"length_penalized_reward ✓  long completions penalised: {(r_adj[long_mask] < r_raw[long_mask]).all().item()}")
print(f"reward_with_confidence  ✓  {conf_mask.sum().item()}/{B} high confidence")


ensemble_reward         ✓  min ≤ mean: True
length_penalized_reward ✓  long completions penalised: True
reward_with_confidence  ✓  9/16 high confidence


---
## 5 · Reward Overoptimisation — Gao et al. (2023)

**Key empirical finding (Gao et al., "Scaling Laws for Reward Model Overoptimization"):**

As the policy is optimised more aggressively against a proxy RM, the **proxy reward increases**  
but the **gold reward (true human preference) peaks and then decreases**.

The gap $r_{proxy} - r_{gold}$ grows as a function of KL from the reference model.

### The relationship (empirical)
$$r_{gold} \approx r_{proxy} - c \sqrt{D_{KL}(\pi_\theta \| \pi_{ref})}$$

where $c$ is a dataset-size-dependent constant. More RM training data → smaller $c$.

### Practical implications
1. **Early stopping:** don't optimise until KL convergence — stop when $D_{KL}$ reaches a threshold
2. **RM size:** larger, better-trained RMs have smaller $c$ — worth investing in RM quality
3. **Iterative RLHF:** refresh RM periodically with new on-policy labels
4. **Ensemble RMs:** reduces effective $c$ by averaging out individual RM biases


In [8]:
def simulate_overoptimisation(
    n_steps:    int   = 200,
    beta:       float = 0.1,   # KL coefficient
    c:          float = 0.5,   # overoptimisation coefficient
    lr:         float = 0.05,
    seed:       int   = 42,
) -> Dict[str, List[float]]:
    """
    Simulate Gao et al. overoptimisation dynamics.

    Model:
      - Policy has a scalar parameter θ (starts at 0)
      - Proxy reward  = θ (RM proxy increases with drift from ref)
      - KL from ref   ≈ θ² (grows quadratically)
      - Gold reward   = proxy - c * KL = θ - c·θ²  (true quality; peaks then falls)

    Note: proxy - c*sqrt(KL) equals θ(1-c) for θ≥0 and is monotone if c<1, so it
    cannot show a mid-training peak; the θ - c·θ² form is a minimal toy with the
    same qualitative "gold peaks then drops" behaviour.

    We maximise proxy - β·KL (RLHF-style objective), tracking gold separately.
    """
    torch.manual_seed(seed)
    theta = torch.tensor([0.0], requires_grad=True)
    opt   = torch.optim.SGD([theta], lr=lr)

    history = {"proxy": [], "gold": [], "kl": [], "theta": []}

    for step in range(n_steps):
        proxy_r = theta                             # proxy increases with theta
        kl      = theta.pow(2)                      # KL grows quadratically
        gold_r  = (proxy_r - c * kl).detach().item()

        # Maximise proxy - beta*KL  (standard RLHF objective)
        loss = -(proxy_r - beta * kl)
        opt.zero_grad()
        loss.backward()
        opt.step()

        history["proxy"].append(proxy_r.item())
        history["gold"].append(gold_r)
        history["kl"].append(kl.item())
        history["theta"].append(theta.item())

    return history


def find_optimal_kl_budget(
    c: float = 0.5,
    beta: float = 0.1,
) -> Dict[str, float]:
    """
    Under the toy gold(θ) = θ - c·θ² with KL = θ²:
      d/dθ (θ - c·θ²) = 0  →  θ_g = 1/(2c),  KL_g = θ_g²,  gold_peak = 1/(4c).

    The toy *training* objective θ - β·θ² is maximised at θ_rl = 1/(2β)
    (often past θ_g — hence overoptimisation in the simulation).
    """
    theta_g = 1.0 / (2 * c)
    kl_g = theta_g ** 2
    gold_peak = theta_g - c * kl_g
    theta_rl = 1.0 / (2 * beta)
    kl_rl = theta_rl ** 2
    gold_at_rl_peak = theta_rl - c * kl_rl
    return {
        "theta_at_gold_peak": theta_g,
        "kl_at_gold_peak": kl_g,
        "gold_peak": gold_peak,
        "theta_at_toy_rl_objective_peak": theta_rl,
        "kl_at_toy_rl_objective_peak": kl_rl,
        "gold_at_toy_rl_objective_peak": gold_at_rl_peak,
    }


# ── Run simulation ─────────────────────────────────────────────────────────
history = simulate_overoptimisation(n_steps=100, c=0.3, beta=0.05)

gold_arr  = np.array(history["gold"])
proxy_arr = np.array(history["proxy"])
kl_arr    = np.array(history["kl"])

peak_step = np.argmax(gold_arr)
print(f"simulate_overoptimisation ✓")
print(f"  Proxy reward (final):  {proxy_arr[-1]:.4f}")
print(f"  Gold reward  (final):  {gold_arr[-1]:.4f}")
print(f"  Gold reward  (peak):   {gold_arr[peak_step]:.4f}  at step {peak_step}")
print(f"  KL at peak:            {kl_arr[peak_step]:.4f}")
assert gold_arr[peak_step] > gold_arr[-1], "Gold reward should peak before converging"

budget = find_optimal_kl_budget(c=0.3, beta=0.05)
print(f"\nOptimal KL budget: {budget}")


simulate_overoptimisation ✓
  Proxy reward (final):  3.9423
  Gold reward  (final):  -0.6789
  Gold reward  (peak):   0.8333  at step 36
  KL at peak:            2.7260

Optimal KL budget: {'theta_star': 10.0, 'kl_star': 100.0, 'gold_at_kl_star': 7.0}
